# Component ablations

Displays the retrieval-stage leave-one-component-out controls. Reranker-retained controls remain pending for the main PC.

**Status:** provisional development evidence; zero locked test queries used.


In [1]:
from pathlib import Path
import csv, hashlib, json, random, statistics
from collections import Counter
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
print(f"Project: {ROOT.name} | fixed seed: {SEED}")


Project: RAABTA_PROJECT_PORTABLE | fixed seed: 20250816


In [2]:
report = json.loads((ROOT / "reports/tables/provisional_retrieval_ablations.json").read_text(encoding="utf-8"))
assert report["test_queries_used"] == 0
for name, values in report["configurations"].items():
    print(f"{name:24s} Recall@10={values['recall_at_10']:.4f} MRR@10={values['mrr_at_10']:.4f}")


full_no_reranker         Recall@10=0.1667 MRR@10=0.0750
no_bm25                  Recall@10=0.1417 MRR@10=0.0529
no_dense                 Recall@10=0.0833 MRR@10=0.0488
no_expansion             Recall@10=0.1583 MRR@10=0.0843
no_fusion                Recall@10=0.0583 MRR@10=0.0232
no_normalization         Recall@10=0.1500 MRR@10=0.0715
no_transliteration       Recall@10=0.0667 MRR@10=0.0253


## Deltas from full system


In [3]:
full = report['configurations']['full_no_reranker']
deltas = []
for name, values in report['configurations'].items():
    if name != 'full_no_reranker':
        deltas.append({'removed': name.removeprefix('no_'), 'delta_R@10': round(values['recall_at_10'] - full['recall_at_10'], 6), 'delta_MRR@10': round(values['mrr_at_10'] - full['mrr_at_10'], 6), 'latency_delta_ms': round(values['mean_latency_ms'] - full['mean_latency_ms'], 3)})
print(json.dumps(sorted(deltas, key=lambda row: row['delta_R@10']), indent=2))


[
  {
    "removed": "fusion",
    "delta_R@10": -0.108334,
    "delta_MRR@10": -0.051773,
    "latency_delta_ms": 63.941
  },
  {
    "removed": "transliteration",
    "delta_R@10": -0.1,
    "delta_MRR@10": -0.049686,
    "latency_delta_ms": -89.442
  },
  {
    "removed": "dense",
    "delta_R@10": -0.083334,
    "delta_MRR@10": -0.026227,
    "latency_delta_ms": -170.487
  },
  {
    "removed": "bm25",
    "delta_R@10": -0.025,
    "delta_MRR@10": -0.022037,
    "latency_delta_ms": -154.161
  },
  {
    "removed": "normalization",
    "delta_R@10": -0.016667,
    "delta_MRR@10": -0.003472,
    "latency_delta_ms": -22.348
  },
  {
    "removed": "expansion",
    "delta_R@10": -0.008334,
    "delta_MRR@10": 0.009352,
    "latency_delta_ms": -70.817
  }
]


## Ablation interpretation


In [4]:
worst = min(deltas, key=lambda row: row['delta_R@10'])
best_mrr = max(report['configurations'].items(), key=lambda item: item[1]['mrr_at_10'])
print('Largest Recall@10 loss:', worst)
print('Highest MRR@10 configuration:', best_mrr[0], best_mrr[1]['mrr_at_10'])
print(report['limitation'])


Largest Recall@10 loss: {'removed': 'fusion', 'delta_R@10': -0.108334, 'delta_MRR@10': -0.051773, 'latency_delta_ms': 63.941}
Highest MRR@10 configuration: no_expansion 0.084339
The six retrieval-component controls isolate retrieval behavior before reranking; the separately measured no-reranking control isolates the reranker. Independent native-speaker review remains pending.


## Interpretation

Outputs above are measurements from frozen local artifacts. Important limitations must remain attached when reused in the report or viva.
